In [ ]:
pip install selenium 
pip install pandas

   ---------------------------------------- 0.0/9.4 MB ? eta -:--:--
   ---------------------------------------- 9.4/9.4 MB 62.9 MB/s  0:00:00

   -- -------------------------------------  1/16 [websocket-client]
   -- -------------------------------------  1/16 [websocket-client]
   -- -------------------------------------  1/16 [websocket-client]
   ----- ----------------------------------  2/16 [urllib3]
   ----- ----------------------------------  2/16 [urllib3]
   ----- ----------------------------------  2/16 [urllib3]
   ----- ----------------------------------  2/16 [urllib3]
   ---------- -----------------------------  4/16 [pysocks]
   ------------ ---------------------------  5/16 [pycparser]
   --------------- ------------------------  6/16 [idna]
   ----------------- ----------------------  7/16 [h11]
   -------------------- -------------------  8/16 [certifi]
   ---------------------- -----------------  9/16 [attrs]
   ------------------------- -------------- 10/16 [wspro

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException,
    ElementClickInterceptedException,
)
import pandas as pd
import time
import re
import os
from openpyxl import load_workbook

# Launch Microsoft Edge browser
driver = webdriver.Edge()
wait = WebDriverWait(driver, 30)

output_file = "Project_History.xlsx"

try:
    # Open website
    driver.get("https://innovation.semperitgroup.com/IdeaManagement/Details/1989")

    print("Please login...")

    # Wait until login completed
    wait.until(EC.url_contains("IdeaManagement"))

    project_ids = range(2025, 2041)

    # Loop through each project ID
    for project_id in project_ids:

        print(f"\nProcessing Project {project_id}")

        try:

            url = f"https://innovation.semperitgroup.com/IdeaManagement/Details/{project_id}"

             # Open the project page
            driver.get(url)
             # Wait until the page body is loaded
            wait.until(
                EC.presence_of_element_located((By.TAG_NAME, "body"))
            )

            # Open History tab
            # Locate the History tab and wait until it is clickable
            history_tab = wait.until(
                EC.element_to_be_clickable(
                    (
                        By.XPATH,
                        "//a[contains(.,'History') or .//span[contains(.,'History')]]"
                    )
                )
            )
             # Scroll the History tab into view
            driver.execute_script(
                "arguments[0].scrollIntoView({block:'center'});",
                history_tab
            )
            # Click the History tab
            history_tab.click()

            time.sleep(2)

            
            # Keep clicking the "Show More" button until it no longer exists
            while True:

                try:
                    # Wait up to 2 seconds for the Show More button
                    show_more = WebDriverWait(driver, 2).until(
                        EC.element_to_be_clickable(
                            (
                                By.XPATH,
                                "//button[contains(.,'Show more')]"
                            )
                        )
                    )
                    # Scroll the Show More button into view
                    driver.execute_script(
                        "arguments[0].scrollIntoView({block:'center'});",
                        show_more
                    )
                    # Small delay before clicking
                    time.sleep(1)

                    try:
                        # Try clicking the button normally
                        show_more.click()
                    except ElementClickInterceptedException:
                        # If normal click fails, click using JavaScript
                        driver.execute_script(
                            "arguments[0].click();",
                            show_more
                        )

                    print("Show More clicked")
                    # Wait for additional history records to load
                    time.sleep(2)

                except TimeoutException:
                     # Exit loop when Show More button is no longer available
                    break
            # Read all visible text from the webpage
            body_text = driver.find_element(By.TAG_NAME, "body").text
            # Split the page text into individual lines
            lines = body_text.splitlines()
             # Create an empty list to store extracted rows
            rows = []
             # Variable to keep track of the current history record
            current = None
            # Regular expression to match history entries
            pattern = re.compile(
                r"^(.*?)\s+(\d{2}/\d{2}/\d{4})\s+(\d{2}:\d{2})\s+(.*)$"
            )
            # Loop through every line on the page
            for line in lines:
                # Check whether the line matches the history pattern
                m = pattern.match(line)

                if m:
                    # Create a dictionary for the main history record
                    current = {
                        "Name": m.group(1).strip(),
                        "Date": m.group(2),
                        "Time": m.group(3),
                        "Event": m.group(4),
                        "Field": "",
                        "Value": ""
                    }
                    # Add the record to the list
                    rows.append(current)
                 # Check for field-value lines (e.g., Status: Approved)
                elif ":" in line and current:

                    field, value = line.split(":", 1)

                    rows.append({
                        "Name": "",
                        "Date": "",
                        "Time": "",
                        "Event": "",
                        "Field": field.strip(),
                        "Value": value.strip()
                    })
             # Convert the extracted rows into a pandas DataFrame
            df = pd.DataFrame(rows)
            # Skip this project if no history was found
            if df.empty:
                print(f"No history found for Project {project_id}")
                continue
            # Insert ProjectID as the first column
            df.insert(0, "ProjectID", project_id)
            # Create a worksheet name using the project ID
            sheet = f"Project_{project_id}"
            # Check if the Excel workbook already exists
            if os.path.exists(output_file):

                try:
                    # Read the existing worksheet
                    existing = pd.read_excel(
                        output_file,
                        sheet_name=sheet
                    )
                    # Append new data to existing data
                    final_df = pd.concat(
                        [existing, df],
                        ignore_index=True
                    )

                except ValueError:
                     # If worksheet doesn't exist, use current DataFrame
                    final_df = df.copy()
                 # Open workbook in append mode
                with pd.ExcelWriter(
                    output_file,
                    engine="openpyxl",
                    mode="a",
                    if_sheet_exists="replace"
                ) as writer:
                     # Replace the worksheet with updated data
                    final_df.to_excel(
                        writer,
                        sheet_name=sheet,
                        index=False
                    )

            else:
                # Create a new workbook if it doesn't exist
                with pd.ExcelWriter(
                    output_file,
                    engine="openpyxl",
                    mode="w"
                ) as writer:
                     # Write the DataFrame to a new worksheet
                    df.to_excel(
                        writer,
                        sheet_name=sheet,
                        index=False
                    )

            print(f"Updated sheet {sheet}")

            print(f"Project {project_id} saved successfully.")

        except Exception as e:

            print(f"Project {project_id} failed : {e}")

    print("\nAll projects completed.")

finally:
    # Close the browser regardless of success or failure
    driver.quit()

Please login...

Processing Project 2025
Project 2025 failed : Message: 
Stacktrace:
	msedgedriver!GetHandleVerifier [0x7ff62dd7ef95+e415]
	msedgedriver!GetHandleVerifier [0x7ff62dd7eff4+e474]
	msedgedriver!GetHandleVerifier [0x7ff62e3f9706+688b86]
	msedgedriver!(No symbol) [0x7ff62d7707a2]
	msedgedriver!(No symbol) [0x7ff62d770a15]
	msedgedriver!(No symbol) [0x7ff62d7ae9c7]
	msedgedriver!(No symbol) [0x7ff62d7673b7]
	msedgedriver!(No symbol) [0x7ff62d7ac688]
	msedgedriver!(No symbol) [0x7ff62d766bfc]
	msedgedriver!(No symbol) [0x7ff62d765e56]
	msedgedriver!(No symbol) [0x7ff62d766a23]
	msedgedriver!(No symbol) [0x7ff62d9a93f1]
	msedgedriver!(No symbol) [0x7ff62d9a580f]
	msedgedriver!(No symbol) [0x7ff62d9b6699]
	msedgedriver!GetHandleVerifier [0x7ff62dd9a681+29b01]
	msedgedriver!GetHandleVerifier [0x7ff62dda2e86+32306]
	msedgedriver!GetHandleVerifier [0x7ff62dd86af4+15f74]
	msedgedriver!GetHandleVerifier [0x7ff62dd86c15+16095]
	msedgedriver!GetHandleVerifier [0x7ff62dd73033+24b3]
	KER